# A4 · Prepare: cleaning, features, and the split by time

**Goal:** turn `raw_emr_export.csv` into model-ready data so that nothing from the future, and nothing recorded after booking, reaches the model.

```
raw ─► 1 duplicates ─► 2 labels ─► 3 age 999 ─► 4 minutes ─► 5 bad targets ─► 6 SPLIT BY TIME ─► 7 fit on train ─► 8 features
       └──────────────────── rules (row by row) ────────────────────┘                          └──── learned ────┘
```

**Before running this notebook,** write the `TODO(you)` functions and make the tests pass:

| File | Your functions | Test |
|---|---|---|
| `slotpred/clean.py` | `normalise_doctor`, `normalise_visit_type`, `fix_age`, `add_minutes`, `target_ok` | `.venv/Scripts/python.exe -m pytest tests/test_clean.py -v` |
| `slotpred/split.py` | `time_split` | `.venv/Scripts/python.exe -m pytest tests/test_split.py -v` |
| `slotpred/features.py` | `add_time_features`, `impute_age`, `fit_prep` | `.venv/Scripts/python.exe -m pytest tests/test_features.py -v` |

Then run the cells below. Sections 4 and 5 are your experiments.

> **A note on the test months.** Sections 4 and 5 look at July–August to *demonstrate* why the split matters. No setting is chosen from these numbers. From A5 onward the test split stays locked until A8.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from slotpred.clean import clean
from slotpred.eda import load_raw
from slotpred.features import FEATURES, fit_prep, make_features, make_xy
from slotpred.split import describe, random_split, time_split

raw = load_raw()
raw.shape

## 1 · Steps 1–5: the rules

`clean()` runs your rules in order and returns a **funnel**: how many rows each step removed.

In [ ]:
df, funnel = clean(raw)
print(f"{len(raw):,} raw rows -> {len(df):,} clean visits")
funnel

**TODO(you): reconcile the funnel with your A3 problem log.** The counts won't all match exactly. For each row, find out why.

| Log # | Problem | A3 count | A4 removed | Same? If not, why? |
|---|---|---|---|---|
| 1 | Duplicates | 175 | | |
| 5 | Missing `consult_end` | 100 | | |
| 7 | Whitefield March–April | 1,753 | | |
| 6 | Under 1 or over 120 min | 669 | | |

*Hint:* the funnel counts a row with two problems only once, under the first reason. Try `(df_before.clinic == "Whitefield") & ...` on the table before step 5. You can rebuild it with `from slotpred.clean import normalise_labels, add_minutes`.

In [ ]:
# TODO(you): explain any difference in the table above with a count or two

## 2 · Step 6: split by time

In [ ]:
splits = time_split(df)
describe(splits)

**Question:** the training split ends on 30 May, not 31 May. Why? *(Hint: `pd.Timestamp("2026-05-31").day_name()`.)*

*Answer:*

In [ ]:
# TODO(you): in which split(s) does Dr Nikhil Verma appear, and how many visits in each?

## 3 · Steps 7–8: fit on train, then build features for every split

`fit_prep` sees **only** `splits["train"]`. `make_features` applies what it learned, unchanged, to all three splits.

In [ ]:
state = fit_prep(splits["train"])
print("age_fill =", state.age_fill)
print({k: len(v) for k, v in state.categories.items()})

X_train, y_train = make_xy(splits["train"], state)
X_valid, y_valid = make_xy(splits["valid"], state)
X_test, y_test = make_xy(splits["test"], state)
display(X_train.head())
X_train.dtypes

In [ ]:
# TODO(you): check two things in X_test
# 1. What happened to Nikhil Verma's rows in the `doctor` column? (compare splits["test"]["doctor"] with X_test["doctor"])
# 2. How many rows have age_missing == 1 in each split, and what is their `age` now?

## 4 · Experiment 1: random split vs split by time

The simplest sensible model: **predict the median `minutes` of the visit's department × visit type**, learned from the training rows. (This is A5's baseline; here it's just a measuring stick.)

The helper is given. You run it on both kinds of split.

In [ ]:
def median_baseline(train, test, key=("department", "visit_type")):
    """Fit group medians on `train`, score on `test`. Returns overall MAE, Dermatology MAE and Dermatology bias."""
    key = list(key)
    medians = train.groupby(key)["minutes"].median().rename("pred")
    t = test.join(medians, on=key)
    t["pred"] = t["pred"].fillna(train["minutes"].median())       # groups never seen in train
    err = t["minutes"] - t["pred"]
    derm = t["department"] == "Dermatology"
    return pd.Series({"MAE": err.abs().mean(), "Derm MAE": err[derm].abs().mean(),
                      "Derm bias (actual - pred)": err[derm].mean(), "test rows": len(t)}).round(2)

In [ ]:
# TODO(you): build this table
#   row "time split":   fit on splits["train"], score on splits["test"]
#   row "random split": r = random_split(df); fit on r["train"], score on r["test"]
#   row "random-split model, scored on Jul-Aug only": fit on r["train"], score on the rows of r["test"] dated >= "2026-07-01"
# hint: pd.DataFrame({"time split": median_baseline(...), ...}).T

**Questions**
1. Which split makes the model look better overall? Which one tells you what will happen after deployment?
2. Dermatology bias: why is it about +3.7 with the split by time and much smaller with the random split? Connect it to the simulator (A2).
3. Rows 2 and 3 are the *same* model. Why is its Dermatology bias so different on the two test sets? So where did the random split's good score come from?
4. Row 3's bias is a little smaller than row 1's. What leaked into the random-split model?

*(Your numbers differ slightly from the lesson's, because the random split depends on its seed.)*

*Answers:*

## 5 · Experiment 2: what is the learned step allowed to see?

Same helper, a finer key: the median **per doctor × visit type**. Fit it two ways and score both on the test months.

In [ ]:
# TODO(you):
# 1. honest: fit on splits["train"]                       -> score on splits["test"]
# 2. leaky:  fit on df (all months, including the test months) -> score on splits["test"]
# 3. The same idea for age: compare splits["train"]["age"].median() with df["age"].median()

**Questions**
1. How much better does the leaky version *look*? Would that improvement exist after deployment?
2. The age median barely moves (or doesn't move at all). Does that make "fit on train only" unnecessary for age? Why or why not?
3. Which kind of statistic leaks the most: ones computed from the inputs (age) or from the target (`minutes`)? Why?

*Answers:*

## 6 · Save the prepared data

One command reruns the whole pipeline and saves the splits and `prep_state.json` for A5 onward. You can also run it from the terminal: `.venv/Scripts/python.exe -m slotpred.prepare`.

In [ ]:
from slotpred import prepare
splits, state = prepare.main()

## Wrap-up questions (answer in `notes/A04-prepare.md` too)

1. Give one example of a **rule** and one of a **learned step** from this lesson. Where in the pipeline may each run?
2. Why is `target_ok` never called by the API, while `normalise_doctor` must be?
3. Why do we keep `doctor` as a feature even though Nikhil Verma is unseen?
4. What would go wrong if you dropped duplicates *after* a random split?